# ARC-AGI-2: conceptual lattice + RDR over ARCGraph objects (symbolic, no neural model)
Frozen prior vocabulary `fill_interior,ray,connect,border,bbox_fill,outline8,L_inertia,D12,S_holes,D4,D1,S_size,P_pixel,P_frame,D10,S_atoms,G_dsl`. Per task, RDR decision lists over lattice concepts are induced
from the training pairs only and executed on the test inputs. Unresolved slots get a 1x1 placeholder,
so any non-zero score comes from the induced rules.





**Probe:** V29 (MDL answer ordering + Fable v3 code lengths + 20 abduced families)

In [ ]:
import hashlib, json, subprocess, sys, zipfile, os
from pathlib import Path
BUNDLE_SHA = '9eedcc404d6015a2c84db58c63ec97c3e55f282faaed5fb9eb479afe0a6ab4fb'
bundles = [p for p in Path('/kaggle/input').rglob('candidate.bundle')]
assert len(bundles) == 1, bundles
payload = bundles[0].parent
assert hashlib.sha256(bundles[0].read_bytes()).hexdigest() == BUNDLE_SHA
cand = Path('/kaggle/working/cand')
if not cand.exists():
    with zipfile.ZipFile(bundles[0]) as z:
        z.extractall(cand)
deps = Path('/kaggle/working/deps')
try:
    import rdflib  # noqa
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--target', str(deps),
                    str(payload / 'rdflib-7.6.0-py3-none-any.whl')], check=True)
challenges = [p for p in Path('/kaggle/input').rglob('arc-agi_test_challenges.json')]
assert len(challenges) == 1, challenges
Path('/kaggle/working/probe').mkdir(exist_ok=True)
CODEX_SHA = '62d4c71378e546093b874b9a1a1ffc61758b85cd531c06b5d4ecab33df18693b'
codex = [p for p in Path('/kaggle/input').rglob('codex_atoms.tgz')]
if codex:
    import tarfile
    assert hashlib.sha256(codex[0].read_bytes()).hexdigest() == CODEX_SHA, 'codex atom bundle digest mismatch'
    cx = Path('/kaggle/working/codex')
    if not cx.exists():
        with tarfile.open(codex[0]) as t: t.extractall(cx)
    os.environ['CODEX_ROOT'] = str(cx); os.environ['S0_REPS'] = str(cx / 's0_reps.json')
print(payload, challenges[0], 'codex' if codex else 'no-codex')


In [ ]:
# Probe payload (zlib + base64): 47 files, written byte-identical to /kaggle/working/probe
import base64 as _b64, json as _json, os as _os, zlib as _zlib
_os.makedirs('/kaggle/working/probe', exist_ok=True)
_P = (
    '...'
)
for _name, _data in _json.loads(_zlib.decompress(_b64.b64decode(_P))).items():
    open(_os.path.join('/kaggle/working/probe', _name), 'wb').write(_b64.b64decode(_data))
print(len(_os.listdir('/kaggle/working/probe')), 'probe files written')


In [ ]:
%%writefile /kaggle/working/predict_m1.py
"""Kaggle driver for the M1b lattice/RDR probe (occupancy2.py).

Every prediction comes from rules induced on the task's own training pairs over the frozen prior
vocabulary; unresolved slots get a 1x1 [[0]] placeholder. Deterministic (sorted iteration), so a
local run and the Kaggle run over the same file must produce the same submission digest.
"""
import argparse, hashlib, json, os, signal, sys, time

PLACEHOLDER = [[0]]


class TaskTimeout(BaseException):
    pass


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--candidate", required=True)      # dir containing image.py / ARCGraph.py
    ap.add_argument("--probe", required=True)          # dir containing occupancy2.py
    ap.add_argument("--vocab", required=True)
    ap.add_argument("--challenges", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--task-seconds", type=int, default=120)
    args = ap.parse_args()
    os.environ["M1B_VOCAB"] = args.vocab
    sys.path[:0] = [args.probe, args.candidate]
    import occupancy2 as probe
    probe.Timeout = TaskTimeout
    ch = json.load(open(args.challenges))
    signal.signal(signal.SIGALRM, lambda *a: (_ for _ in ()).throw(TaskTimeout()))
    sub, solved_slots = {}, 0
    for k in sorted(ch):
        task = ch[k]
        slots = [{"attempt_1": PLACEHOLDER, "attempt_2": PLACEHOLDER} for _ in task["test"]]
        try:
            signal.alarm(args.task_seconds)
            attempts, _, _ = probe.solve(task)
            for rank, a in enumerate(attempts[:2]):
                for i, g in enumerate(a["preds"]):
                    if g and all(isinstance(r, list) and r for r in g):
                        slots[i]["attempt_1" if rank == 0 else "attempt_2"] = g
        except TaskTimeout:
            pass
        except Exception:
            pass
        finally:
            signal.alarm(0)
        sub[k] = slots
    blob = json.dumps(sub, sort_keys=True).encode()
    open(args.out, "w").write(json.dumps(sub))
    print(json.dumps({"tasks": len(sub), "digest": hashlib.sha256(blob).hexdigest()}))


if __name__ == "__main__":
    main()


In [ ]:
# Parity gate: the notebook must replicate the local result on the public 120-task evaluation set.
PROBE_SHA = '77158c8769c21315b77d373609e1f9017435f8f048991cc2704fd5f495794320'
VOCAB = 'fill_interior,ray,connect,border,bbox_fill,outline8,L_inertia,D12,S_holes,D4,D1,S_size,P_pixel,P_frame,D10,S_atoms,G_dsl'
EVAL_SHA = 'e7c62a4bd211867c6b538f66b8013b81f299663c82ca062f49a52bf439d6e4e8'
EXPECTED_DIGEST = '277d8cef332ac1b8e1114fc98d3c9f32035c2b91befb3ec6f98b377d0157ff12'
EXPECTED_CORRECT = 52
assert hashlib.sha256(Path('/kaggle/working/probe/occupancy2.py').read_bytes()).hexdigest() == PROBE_SHA
evals = [p for p in Path('/kaggle/input').rglob('arc-agi_evaluation_challenges.json')
         if hashlib.sha256(p.read_bytes()).hexdigest() == EVAL_SHA]
assert evals, 'identical public evaluation file not found; parity cannot be verified'
sols = list(evals[0].parent.glob('arc-agi_evaluation_solutions.json'))
env = dict(os.environ, PYTHONHASHSEED='0', OMP_NUM_THREADS='1', MPLBACKEND='Agg',
           PYTHONPATH=os.pathsep.join([str(deps), os.environ.get('PYTHONPATH', '')]))
def run(ch, out):
    subprocess.run([sys.executable, '/kaggle/working/predict_m1.py', '--candidate', str(cand),
                    '--probe', '/kaggle/working/probe', '--vocab', VOCAB, '--challenges', str(ch),
                    '--out', out, '--task-seconds', '300'], check=True, env=env)
    return json.load(open(out))
ev = run(evals[0], '/kaggle/working/parity_eval.json')
digest = hashlib.sha256(json.dumps(ev, sort_keys=True).encode()).hexdigest()
correct = None
if sols:
    ans = json.load(open(sols[0]))
    correct = sum(any(ev[k][i][a] == t for a in ('attempt_1', 'attempt_2')) for k in ans for i, t in enumerate(ans[k]))
report = {'eval_file': str(evals[0]), 'prediction_digest': digest, 'digest_match': digest == EXPECTED_DIGEST,
          'correct_of_172': correct, 'expected_correct': EXPECTED_CORRECT}
json.dump(report, open('/kaggle/working/parity_report.json', 'w'), indent=1)
print(report)
assert digest == EXPECTED_DIGEST, 'PARITY FAILED: predictions differ from the frozen local run'
assert correct in (None, EXPECTED_CORRECT), f'PARITY FAILED: expected {EXPECTED_CORRECT} correct, got {correct}'


In [ ]:
sub = run(challenges[0], '/kaggle/working/submission.json')
src = json.load(open(challenges[0]))
assert set(sub) == set(src) and all(len(sub[k]) == len(src[k]['test']) for k in src)
print(len(sub), 'tasks written')
